In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 59.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=011f047a357f8fd2297d8eb9fc08766df745fc5f0ba906ed862dae8da0628ac0
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


Implement a demonstration of the BB84 protocol, with and without an at-
tacker. Although BB84 is a communication protocol involving two separate
agents (or three, including an attacker), implement it as a single sequential
program — but make it clear which parts of the code correspond to each
agent. To generate the random choices that are needed by the protocol,
measure a suitable quantum state such as 1√2 (|0⟩ + |1⟩). Don’t use a stan-
dard Python random number generator. The attacker should try to obtain
information about the key, somehow, for example by measuring some of the
qubits. You can also try other kinds of attack. The participants in the
protocol should report an attack by using some threshold on the proportion
of disrupted qubits

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [2]:
# ─────────────────────────────────────────────────────────────
#  QUANTUM RANDOM NUMBER GENERATOR
#
#  Prepares |+> = 1/sqrt(2) (|0> + |1>) by applying H to |0>,
#  then measures. Each shot yields 0 or 1 with probability 1/2.
#  This is the ONLY source of randomness in the entire protocol.
# ─────────────────────────────────────────────────────────────
def quantum_random_bits(n: int) -> list:
    """Generate n quantum-random bits by measuring |+> = H|0>."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # |0>  ->  1/sqrt(2)(|0> + |1>)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=n)
    counts = job.result().get_counts()
    bits = []
    for bitstring, freq in counts.items():
        bits.extend([int(bitstring)] * freq)
    return bits[:n]


In [3]:
# ─────────────────────────────────────────────────────────────
#  ENCODING AND MEASUREMENT
#
#  Basis 0 = rectilinear {|0>, |1>}   (+)
#  Basis 1 = diagonal    {|+>, |->}   (x)
#
#  bit=0, basis=0  ->  |0>
#  bit=1, basis=0  ->  |1>
#  bit=0, basis=1  ->  |+> = H|0>
#  bit=1, basis=1  ->  |-> = HX|0>
# ─────────────────────────────────────────────────────────────

# ── ALICE ────────────────────────────────────────────────────
def alice_encode(bit: int, basis: int) -> QuantumCircuit:
    """
    ALICE encodes one classical bit into a qubit.
    Returns a QuantumCircuit with the qubit prepared in the
    correct polarisation state.
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)        # |0> -> |1>
    if basis == 1:
        qc.h(0)        # rotate into diagonal basis
    return qc

# ── BOB / EVE ────────────────────────────────────────────────
def measure_in_basis(qc: QuantumCircuit, basis: int) -> int:
    """
    BOB (or EVE) measures a qubit in the chosen basis.
    Copies the circuit, appends a basis rotation if needed,
    then measures. Returns 0 or 1.
    """
    qc = qc.copy()
    if basis == 1:
        qc.h(0)        # rotate back from diagonal basis
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    counts = job.result().get_counts()
    return int(list(counts.keys())[0])


# =============================================================
#  SCENARIO 1 — BB84 WITH NO ATTACKER
# =============================================================
def run_bb84_no_attacker(n_bits: int = 100, error_threshold: float = 0.15):
    """
    Honest BB84 between Alice and Bob only.

    Steps:
      1. ALICE  picks random bits + bases, encodes qubits
      2. BOB    picks random bases, measures each qubit
      3. PUBLIC  Alice & Bob compare bases over classical channel
      4. SIFT   Keep only bits where both chose the same basis
      5. CHECK  Sacrifice ~20% of sifted key to estimate QBER
      6. REPORT Flag attack if QBER > threshold
    """
    print("=" * 58)
    print("  SCENARIO 1 — BB84, no attacker")
    print("=" * 58)

    # ── Step 1: ALICE prepares ───────────────────────────────
    print(f"\n[ALICE] Preparing {n_bits} qubits ...")
    alice_bits  = quantum_random_bits(n_bits)
    alice_bases = quantum_random_bits(n_bits)
    qubits = [alice_encode(b, basis)
               for b, basis in zip(alice_bits, alice_bases)]

    print(f"  bits  (first 10): {alice_bits[:10]}")
    print(f"  bases (first 10): {alice_bases[:10]}  (0=+, 1=x)")

    # ── Step 2: BOB measures ─────────────────────────────────
    print(f"\n[BOB]   Measuring {n_bits} qubits ...")
    bob_bases   = quantum_random_bits(n_bits)
    bob_results = [measure_in_basis(qc, basis)
                   for qc, basis in zip(qubits, bob_bases)]

    print(f"  bases   (first 10): {bob_bases[:10]}")
    print(f"  results (first 10): {bob_results[:10]}")

    # ── Step 3: PUBLIC basis reconciliation ──────────────────
    matching = [i for i in range(n_bits) if alice_bases[i] == bob_bases[i]]
    print(f"\n[PUBLIC] Matching bases: {len(matching)} / {n_bits}")

    # ── Step 4: SIFT ─────────────────────────────────────────
    alice_key = [alice_bits[i]   for i in matching]
    bob_key   = [bob_results[i]  for i in matching]
    print(f"\n[SIFT]  Sifted key length: {len(alice_key)} bits")

    # ── Step 5: ERROR ESTIMATION ─────────────────────────────
    check_n     = max(1, len(alice_key) // 5)
    errors      = sum(a != b for a, b in zip(alice_key[:check_n],
                                              bob_key[:check_n]))
    qber        = errors / check_n
    final_alice = alice_key[check_n:]
    final_bob   = bob_key[check_n:]

    print(f"\n[CHECK] Sacrificing {check_n} bits for error estimation ...")
    print(f"  Errors: {errors} / {check_n}  ->  QBER = {qber:.2%}")
    print(f"  Threshold: {error_threshold:.0%}")

    # ── Step 6: REPORT ───────────────────────────────────────
    print("\n[RESULT]")
    if qber > error_threshold:
        print("  WARNING: ATTACK DETECTED -- aborting key exchange.")
    else:
        print("  OK: Channel is secure.")
        print(f"  OK: Shared key length: {len(final_alice)} bits")
        match = all(a == b for a, b in zip(final_alice, final_bob))
        print(f"  OK: Keys identical: {match}")

    return {
        "qber": qber,
        "sifted_len": len(alice_key),
        "key_len": len(final_alice),
        "attack_detected": qber > error_threshold,
    }

In [5]:
# =============================================================
#  SCENARIO 2 — BB84 WITH EVE (intercept-resend attack)
# =============================================================
def run_bb84_with_attacker(n_bits: int = 100,
                            error_threshold: float = 0.15,
                            eve_intercept_prob: float = 1.0):
    """
    BB84 with Eve performing an intercept-resend attack.

    Eve's strategy (per qubit):
      - With probability eve_intercept_prob, Eve intercepts the qubit.
      - She picks a random basis and measures (collapsing the state).
      - She re-encodes her result in the same basis she used.
      - She forwards the new (possibly disturbed) qubit to Bob.

    Why this introduces errors:
      - When Eve's basis != Alice's (prob 1/2), the qubit is
        projected into the wrong basis.
      - Bob then gets the wrong bit ~50% of the time in those
        cases, even when his basis matches Alice's.
      - Expected QBER from full intercept: 0.5 x 0.5 = 25%.
    """
    print("=" * 58)
    print(f"  SCENARIO 2 -- BB84, Eve intercepts {eve_intercept_prob:.0%} of qubits")
    print("=" * 58)

    # ── ALICE ────────────────────────────────────────────────
    print(f"\n[ALICE] Preparing {n_bits} qubits ...")
    alice_bits  = quantum_random_bits(n_bits)
    alice_bases = quantum_random_bits(n_bits)
    qubits      = [alice_encode(b, basis)
                   for b, basis in zip(alice_bits, alice_bases)]

    # ── EVE (attacker) ───────────────────────────────────────
    print(f"\n[EVE]   Running intercept-resend attack on quantum channel ...")
    eve_bases      = quantum_random_bits(n_bits)
    # Use quantum randomness to decide which qubits Eve intercepts
    intercept_roll = quantum_random_bits(n_bits)
    intercept_mask = [r < eve_intercept_prob for r in intercept_roll]

    forwarded_qubits = []
    n_intercepted = 0

    for i in range(n_bits):
        if intercept_mask[i]:
            n_intercepted += 1
            # Eve measures Alice's qubit in her random basis
            eve_bit = measure_in_basis(qubits[i], eve_bases[i])
            # Eve re-encodes her result and sends it on to Bob
            forwarded_qubits.append(alice_encode(eve_bit, eve_bases[i]))
        else:
            # Qubit passes through unmodified
            forwarded_qubits.append(qubits[i])

    print(f"  Qubits intercepted: {n_intercepted} / {n_bits}")

    # ── BOB ──────────────────────────────────────────────────
    print(f"\n[BOB]   Measuring received (possibly disturbed) qubits ...")
    bob_bases   = quantum_random_bits(n_bits)
    bob_results = [measure_in_basis(qc, basis)
                   for qc, basis in zip(forwarded_qubits, bob_bases)]

    # ── PUBLIC basis reconciliation ──────────────────────────
    matching  = [i for i in range(n_bits) if alice_bases[i] == bob_bases[i]]

    # ── SIFT ─────────────────────────────────────────────────
    alice_key = [alice_bits[i]  for i in matching]
    bob_key   = [bob_results[i] for i in matching]

    # ── ERROR ESTIMATION ─────────────────────────────────────
    check_n = max(1, len(alice_key) // 5)
    errors  = sum(a != b for a, b in zip(alice_key[:check_n],
                                          bob_key[:check_n]))
    qber    = errors / check_n

    print(f"\n[CHECK] QBER = {errors}/{check_n} = {qber:.2%}")
    print(f"        Threshold = {error_threshold:.0%}")
    print(f"        (Theoretical QBER for 100% intercept ~= 25%)")

    # ── REPORT ───────────────────────────────────────────────
    print("\n[RESULT]")
    if qber > error_threshold:
        print("  WARNING: ATTACK DETECTED -- aborting key exchange.")
    else:
        print("  NOTE: Attack NOT detected (Eve was lucky or intercept rate low).")
        print(f"  Key length: {len(alice_key) - check_n} bits  <- potentially compromised!")

    return {
        "qber": qber,
        "n_intercepted": n_intercepted,
        "sifted_len": len(alice_key),
        "attack_detected": qber > error_threshold,
    }

In [6]:
# =============================================================
#  MAIN — run all three scenarios
# =============================================================
N         = 100    # number of qubits
THRESHOLD = 0.15   # 15% QBER threshold for attack detection

# Scenario 1: honest channel
r1 = run_bb84_no_attacker(n_bits=N, error_threshold=THRESHOLD)

print("\n")

# Scenario 2: Eve intercepts every qubit (expect QBER ~25% -> detected)
r2 = run_bb84_with_attacker(n_bits=N,
                             error_threshold=THRESHOLD,
                             eve_intercept_prob=1.0)

print("\n")

# Scenario 3: Eve intercepts half the qubits (expect QBER ~12.5% -> borderline)
r3 = run_bb84_with_attacker(n_bits=N,
                             error_threshold=THRESHOLD,
                             eve_intercept_prob=0.5)

# Summary table
print("\n" + "=" * 58)
print("  SUMMARY")
print("=" * 58)
print(f"{'Scenario':<35} {'QBER':>6}  {'Detected':>10}")
print("-" * 58)
print(f"{'No attacker':<35} {r1['qber']:>6.2%}  {'No':>10}")
print(f"{'Eve 100% intercept':<35} {r2['qber']:>6.2%}  {str(r2['attack_detected']):>10}")
print(f"{'Eve 50% intercept':<35} {r3['qber']:>6.2%}  {str(r3['attack_detected']):>10}")
print("=" * 58)


  SCENARIO 1 — BB84, no attacker

[ALICE] Preparing 100 qubits ...
  bits  (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  bases (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  (0=+, 1=x)

[BOB]   Measuring 100 qubits ...
  bases   (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  results (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

[PUBLIC] Matching bases: 100 / 100

[SIFT]  Sifted key length: 100 bits

[CHECK] Sacrificing 20 bits for error estimation ...
  Errors: 0 / 20  ->  QBER = 0.00%
  Threshold: 15%

[RESULT]
  OK: Channel is secure.
  OK: Shared key length: 80 bits
  OK: Keys identical: True


  SCENARIO 2 -- BB84, Eve intercepts 100% of qubits

[ALICE] Preparing 100 qubits ...

[EVE]   Running intercept-resend attack on quantum channel ...
  Qubits intercepted: 45 / 100

[BOB]   Measuring received (possibly disturbed) qubits ...

[CHECK] QBER = 0/19 = 0.00%
        Threshold = 15%
        (Theoretical QBER for 100% intercept ~= 25%)

[RESULT]
  NOTE: Attack NOT detected (Eve was lucky or 